# NB3 · Measure — the oracle sweep

Turns trained backbones into **per-sample MSC tables**, which are the
scientific artifact of the whole project.

For every run:

1. **Train exit heads** — K linear heads on the frozen backbone. Freezing is not
   an optimisation, it is the definition: if the backbone adapted while the
   heads trained, each exit would read a *different* network and the "same model
   under reduced compute" interpretation collapses.
2. **Sweep every configuration on every sample** — depth (K exits), resolution
   (5 native + 5 proxy), precision (5). No early-exit shortcut: the
   stable-sufficiency definition quantifies over *all larger* budgets, so
   stopping at the first agreement would record exactly the accidental early
   agreement the definition exists to reject.
3. **Compute the difficulty battery** and prediction depth.
4. **Write** `per_sample/test.parquet` and `per_sample/train_holdout.parquet`.

Inference-only and idempotent — ~1 GPU-h per run, and re-running skips anything
already measured.

## Why `train_holdout` exists

EL2N and forgetting-events are **training-set** quantities, undefined on any
split the model never trained on. Running Q4 without them handicaps the
difficulty battery, which flatters MSC — that is exactly the defect that
inflated the CIFAR ΔR² by 2.5× and had to be withdrawn.

`train_holdout` is 15,000 training images evaluated with augmentation off. It is
not held out of training.

## The coverage alarm at the end is not decoration

On CIFAR, six runs were trained and never measured — the cheapest architectures,
which the scheduler places last. One architecture ended up with **zero** measured
seeds, contributed nothing to any analysis, and the atlas was 14 architectures
while every document said 15. A noise ceiling needs **two** measured seeds
minimum.

In [ ]:
# ============================================================================
# CELL 1 -- unpack the library.  Runs in every notebook.  No network.
# ============================================================================
# Writes two files into the working directory and imports them:
#
#   msc_lib.py    df12d523fefe   the pipeline: data, zoo, training, measurement
#   msc_core.py   2cc4ba5e0935   the reference maths: the MSC definition and
#                                    every statistic in the paper
#
# Both are GENERATED from src/ by build_notebooks_in100.py. Editing the base64
# below does nothing that survives a rebuild -- edit src/msc_lib.py instead.
#
# NOTHING IS INSTALLED HERE. This pipeline runs offline; the packages must
# already be present (see requirements.txt). A missing one is reported by name
# with what it costs you, rather than silently pip-installing on a machine that
# may have no network.
import base64, os, sys
from pathlib import Path

# Offline guards must be set BEFORE anything that might fetch is imported.
os.environ.setdefault('MSC_OFFLINE', '1')

WORK = Path.cwd()
_LIB = (
    '...',
    'PSAiZGYxMmQ1MjNmZWZlIgo=',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]          # force reimport if this cell is re-run
import importlib
importlib.invalidate_caches()

_MISSING = []
for _pkg, _why in (('torch', 'everything'),
                   ('torchvision', 'resnet/vgg/shufflenet/swin'),
                   ('numpy', 'everything'), ('pandas', 'every table'),
                   ('pyarrow', 'per_sample/*.parquet -- the science'),
                   ('yaml', 'config.yaml per run'),
                   ('scipy', 'Spearman = Q1 and Q3'),
                   ('sklearn', 'Q4 delta-R2, Q2 PCA'),
                   ('psutil', 'host telemetry columns'),
                   ('pynvml', 'GPU power -- energy columns are NA without it'),
                   ('fvcore', 'FLOPs. rho is DEFINED in FLOPs.')):
    try:
        __import__(_pkg)
    except ImportError:
        _MISSING.append(f'{_pkg:12s} {_why}')
if _MISSING:
    print('MISSING PACKAGES -- install these, then restart the kernel:')
    for _m in _MISSING:
        print('   ', _m)
    raise SystemExit('see requirements.txt')

import msc_lib as M
import torch

# D-62. Prove the module that LOADED is the module that SHIPPED.
#
# Twice now a fix was applied, verified, regenerated -- and the run failed with
# the identical error, because the code executing was not the code on disk.
# Jupyter keeps an imported module until something removes it, and any object
# built from the old module (a Session, say) keeps its old functions even after
# a reimport. There was no mechanism that could tell the difference, so the
# evidence looked like "the fix does not work" when it was "the fix never ran".
#
# Rule 5: a cache must answer "is what I have still VALID", not "do I have
# something". The stamp is written into the bytes this cell decodes, so it
# cannot drift from them.
_want = 'df12d523fefe'
_got = getattr(M, '__MSC_BUILD__', None)
if _got != _want:
    raise RuntimeError(
        f"STALE msc_lib: this notebook ships build {_want} but the imported "
        f"module reports {_got}.
"
        f"  loaded from: {getattr(M, '__file__', '?')}
"
        f"  Restart the kernel (Kernel -> Restart) and run all cells. Objects "
        f"created before a reimport keep the OLD code even after this cell "
        f"rewrites the file (D-62).")
# D-68. Is this NOTEBOOK current with the repository?
#
# The check above proves the module matches the notebook. It CANNOT catch a
# stale notebook, because both sides come from the same .ipynb -- they always
# agree with each other and can be arbitrarily old together.
#
# Jupyter saves an open notebook on run. So regenerating NB3 on disk while it
# sits open in a tab means the tab's copy wins the moment you run it: the fixed
# notebook is silently replaced by the one that was open, and the fix appears
# not to have been applied. That happened here -- NB3 was regenerated with
# `done_fn=sess.measured, stage='measure'`, and the version that ran had
# neither.
#
# The repository source is the authority. If it has moved on, this notebook is
# stale and must be reopened, not re-run.
_repo = WORK.parent / 'src' / 'msc_lib.py'
if _repo.exists():
    import hashlib as _h
    _repo_sha = _h.sha256(_repo.read_bytes()).hexdigest()[:12]
    if _repo_sha != _want:
        raise RuntimeError(
            f"STALE NOTEBOOK: this file embeds msc_lib {_want}, but "
            f"src/msc_lib.py is {_repo_sha}.
"
            f"  You are running an older copy of this notebook. Jupyter saves "
            f"an open notebook when you run it, so an open tab silently "
            f"overwrites a regenerated file.
"
            f"  FIX: close this notebook WITHOUT saving, run "
            f"`python build_notebooks_in100.py`, then reopen it (D-68).")
    print(f'msc_lib build {_got} verified, and current with src/')
else:
    print(f'msc_lib build {_got} verified (repo source not visible)')

print(f'msc_lib {M.__version__}   torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name}  {_p.total_memory/2**30:.1f} GiB  sm_{_p.major}{_p.minor}')
else:
    print('  *** NO CUDA. A CPU-only torch trains at roughly 1/200th speed')
    print('  *** while reporting entirely plausible numbers. Fix this first.')

In [ ]:
# ============================================================================
# CELL 2 -- WHERE EVERYTHING LIVES
# ============================================================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine gets `msc_data/in100` and `msc_results`.
#
# The previous version defaulted to r'D:\msc_data\in100'. There is no D:
# drive here, and the failure was
#
#     FileNotFoundError: [WinError 3] The system cannot find the path
#     specified: 'D:\'
#
# forty lines deep inside pathlib, naming neither the setting nor the file that
# had to change. A default that names a drive letter is wrong on any machine
# without that letter (D-44).
#
# Set them explicitly if you want somewhere specific. Both are checked below by
# WRITING A PROBE FILE AND READING IT BACK -- os.access lies on Windows shares.
#
#   data     ~26 GB   the packed dataset, read-only after NB1
#   results ~120 GB   every run. Nothing here is ever deleted.

DATA_DIR = None      # e.g. r'E:\msc_data\in100'   -- None = choose for me
MSC_ROOT = None      # e.g. r'E:\msc_results'        -- None = choose for me

# ---------------------------------------------------------------------------
import os

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_IN100_DIR'] = DATA_DIR
os.environ['MSC_SCRATCH'] = MSC_ROOT

# None of the analysis notebooks may hardcode a phase: whichever one
# NB2 actually trained is the one to read (D-65). Set PHASE by hand
# below to override.
PHASE = M.detect_phase(MSC_ROOT, prefer='p1')
print(f'phase: {PHASE}   on disk: {M.phases_present(MSC_ROOT)}')
sess = M.Session(account='local', phase=PHASE, dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 worker_id=0, num_workers=1)

print()
print('layout under MSC_ROOT:')
print('  runs/{run_id}/  config.yaml  summary.json  STATUS.json')
print('                   metrics/     epochs.csv  final.csv  confusion_matrix.csv')
print('                                per_class.csv  exit_metrics.csv')
print('                   telemetry/   energy_samples.csv  system_samples.csv')
print('                                step_traces.jsonl')
print('                   per_sample/  test.parquet  train_holdout.parquet')
print('                                train_dynamics.parquet  meta.json')
print('                   checkpoints/ ckpt_last.pt  ckpt_best.pt')
print('                   env/         environment.json')
print('                   exit_heads.pt')
print('  budgets/{arch}.json     FLOPs per compute configuration')
print('  registry/events/*.jsonl  what ran, when, and how it ended')
print('  analysis/                Q1-Q4 outputs')
print('  tables/  paper/figures/  console/')

In [ ]:
# PHASE and `sess` come from the paths cell above, which DETECTS the phase
# that actually has runs rather than naming one (D-65).
sess.repair_ledger()

trained = [r['run_id'] for r in sess.completed_runs(phase=PHASE)]
todo    = [r for r in trained if not sess.measured(r)]

print(f'{len(trained)} trained run(s), {len(todo)} still to measure')
for r in todo:
    print(f'  {r}')
if not todo and trained:
    print('  everything already measured -- this notebook is a no-op')

In [ ]:
cfgs = [sess.config(M.parse_run_id(r)['arch'], seed=M.parse_run_id(r)['seed'])
        for r in todo]
# done_fn/stage are NOT optional here. Without them plan_work asks
# "is it trained?" to decide whether to measure, skips every run, and
# reports success having done nothing (D-67).
results = sess.run_all(cfgs, fn=sess.oracle, title='measurement',
                       done_fn=sess.measured, stage='measure')

for r in results:
    print(f"  {r.get('status','?'):9s} {r['run_id']}")

---
## Coverage — and the alarm

In [ ]:
import collections
per_arch = collections.defaultdict(lambda: {'trained': 0, 'measured': 0})
for r in sess.completed_runs(phase=PHASE):
    a = M.parse_run_id(r['run_id'])['arch']
    per_arch[a]['trained'] += 1
    per_arch[a]['measured'] += int(sess.measured(r['run_id']))

print(f"{'arch':18s} {'trained':>8s} {'measured':>9s}   status")
weak = []
for a in M.zoo_for_dataset('imagenet100'):
    t, m = per_arch[a]['trained'], per_arch[a]['measured']
    flag = 'OK' if m >= 2 else ('ONE SEED -- no ceiling possible' if m == 1
                                else 'NOTHING -- contributes to no analysis')
    if m < 2:
        weak.append(a)
    print(f'{a:18s} {t:8d} {m:9d}   {flag}')

print()
if weak:
    print(f'  *** ALARM: {len(weak)} architecture(s) have fewer than 2 measured')
    print(f'  *** seeds: {weak}')
    print('  *** A noise ceiling needs two. These contribute to NOTHING --')
    print('  *** not Q1, not Q3, not Q4 -- and any claim about "eight')
    print('  *** architectures" is false until this is closed.')
else:
    print('  every architecture has >= 2 measured seeds. Ceilings are computable.')

In [ ]:
status = sess.confirm_on_disk([r['run_id'] for r in sess.completed_runs(phase=PHASE)],
                              measured=True)
print()
print('Next: NB4_Analysis (CPU only, minutes).' if not status['at_risk']
      else 'Fix the AT RISK runs before analysing.')